In [1]:
#Script to download the dataset from Kaggle and extract it to the data folder
#Uncomment and run once

"""
import zipfile
import os

!kaggle auth login
!kaggle datasets download irkaal/foodcom-recipes-and-reviews
with zipfile.ZipFile('foodcom-recipes-and-reviews.zip', 'r') as zip_ref:
        zip_ref.extractall('./data')
os.remove('foodcom-recipes-and-reviews.zip')
"""

"\nimport zipfile\nimport os\n\n!kaggle auth login\n!kaggle datasets download irkaal/foodcom-recipes-and-reviews\nwith zipfile.ZipFile('foodcom-recipes-and-reviews.zip', 'r') as zip_ref:\n        zip_ref.extractall('./data')\nos.remove('foodcom-recipes-and-reviews.zip')\n"

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import AgglomerativeClustering
from sentence_transformers import SentenceTransformer
from scipy.sparse import save_npz
import joblib
import pandas as pd
import re
import numpy as np

In [3]:
df=pd.read_csv('data/recipes.csv')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 522517 entries, 0 to 522516
Data columns (total 28 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   RecipeId                    522517 non-null  int64  
 1   Name                        522517 non-null  str    
 2   AuthorId                    522517 non-null  int64  
 3   AuthorName                  522517 non-null  str    
 4   CookTime                    439972 non-null  str    
 5   PrepTime                    522517 non-null  str    
 6   TotalTime                   522517 non-null  str    
 7   DatePublished               522517 non-null  str    
 8   Description                 522512 non-null  str    
 9   Images                      522516 non-null  str    
 10  RecipeCategory              521766 non-null  str    
 11  Keywords                    505280 non-null  str    
 12  RecipeIngredientQuantities  522514 non-null  str    
 13  RecipeIngredientParts    

In [4]:
target_columns = [
    'RecipeId', 'Name', 'Images', 'RecipeCategory', 'Keywords',
    'RecipeIngredientParts', 'AggregatedRating', 'ReviewCount',
    'TotalTime', 'Calories', 'RecipeInstructions'
]
df=df[target_columns].copy()

def parse_r_array(text):
    if pd.isna(text):
        return ""

    items = re.findall(r'"([^"]*)"', str(text))
    return " ".join(items)

df['ingredients_clean'] = df['RecipeIngredientParts'].apply(parse_r_array)
df['keywords_clean'] = df['Keywords'].apply(parse_r_array)

df['features_text'] = (
    df['RecipeCategory'].fillna('') + ' ' + 
    df['keywords_clean'] + ' ' + 
    df['ingredients_clean']
)

In [5]:
print(df.head().T)
df.info()

                                                                       0  \
RecipeId                                                              38   
Name                                   Low-Fat Berry Blue Frozen Dessert   
Images                 c("https://img.sndimg.com/food/image/upload/w_...   
RecipeCategory                                           Frozen Desserts   
Keywords               c("Dessert", "Low Protein", "Low Cholesterol",...   
RecipeIngredientParts  c("blueberries", "granulated sugar", "vanilla ...   
AggregatedRating                                                     4.5   
ReviewCount                                                          4.0   
TotalTime                                                       PT24H45M   
Calories                                                           170.9   
RecipeInstructions     c("Toss 2 cups berries with sugar.", "Let stan...   
ingredients_clean      blueberries granulated sugar vanilla yogurt le...   
keywords_cle

In [6]:
tfidf_baseline = TfidfVectorizer(
    max_features=15000, 
    analyzer='word', 
    ngram_range=(1, 2), 
    stop_words='english'
)
X_baseline = tfidf_baseline.fit_transform(df['features_text'])

In [7]:
nn_baseline_temp = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
nn_baseline_temp.fit(X_baseline)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [8]:
cv = CountVectorizer(stop_words='english', token_pattern=r'(?u)\b\w+\b') 
cv_matrix = cv.fit_transform(df['ingredients_clean'])
vocab = cv.get_feature_names_out()
word_freqs = np.array(cv_matrix.sum(axis=0)).flatten()
df_vocab = pd.DataFrame({'ingredient': vocab, 'freq': word_freqs})
print(f"found {len(df_vocab)} unique words.")

found 3008 unique words.


In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2') 
X_bert = model.encode(df_vocab['ingredient'].tolist(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

In [10]:
clustering = AgglomerativeClustering(
    n_clusters=None, 
    distance_threshold=0.15, 
    metric='cosine',
    linkage='average'
)
df_vocab['cluster_id'] = clustering.fit_predict(X_bert)

In [11]:
#block for merging synonyms
synonym_mapping = {}
grouped = df_vocab.groupby('cluster_id')

for cluster_id, group in grouped:
    if len(group) == 1:
        continue
        
    leader_idx = group['freq'].idxmax()
    leader_word = group.loc[leader_idx, 'ingredient']
    
    for _, row in group.iterrows():
        current_word = row['ingredient']
        if current_word != leader_word:
            synonym_mapping[current_word] = leader_word

print(f"found {len(synonym_mapping)} synonyms for merging.")

found 272 synonyms for merging.


In [12]:
def apply_synonyms(text):
    if not isinstance(text, str): 
        return text
    words = text.split()
    # 
    return " ".join([synonym_mapping.get(w, w) for w in words])

df['ingredients_bert_clean'] = df['ingredients_clean'].apply(apply_synonyms)

In [13]:
df['features_text_final'] = (
    df['RecipeCategory'].fillna('') + " " + 
    df['keywords_clean'].fillna('') + " " + 
    df['ingredients_bert_clean'].fillna('')
)

In [14]:
tfidf = TfidfVectorizer(max_features=15000, analyzer='word', ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(df['features_text_final'])

In [15]:
nn_model = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
nn_model.fit(X_tfidf)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [16]:
print("synonyms:")
count = 0
for old_word, new_word in synonym_mapping.items():
    old_freq = df_vocab[df_vocab['ingredient'] == old_word]['freq'].values[0]
    # only print synonyms that occur more than 50 times
    if old_freq > 50: 
        print(f"[{old_freq} times] {old_word}  --->  {new_word}")
        count += 1
    if count >= 20:
        break

synonyms:
[2868 times] boiled  --->  boiling
[1052 times] slice  --->  sliced
[1108 times] slices  --->  sliced
[16002 times] powdered  --->  powder
[123 times] courgette  --->  courgettes
[56 times] roll  --->  rolled
[95 times] flake  --->  flaked
[283 times] pod  --->  pods
[188 times] tail  --->  tails
[141 times] bag  --->  bags
[737 times] stewing  --->  stewed
[482 times] lentil  --->  lentils
[60 times] crayfish  --->  crawfish
[88 times] bars  --->  bar
[27855 times] dry  --->  dried
[1269 times] mangoes  --->  mango
[966 times] pear  --->  pears
[1750 times] rib  --->  ribs
[538 times] legs  --->  leg
[10784 times] chilies  --->  chili


In [17]:
user_liked_indices = [
    3735,   
    454645,  
    1284, 
    45647, 
    45435,  
    5007,  
    8000,  
    78906,
    165870, 
    543,
    543,
    78664,
    98,
    7654 
]

In [18]:
# single recipe test
def test_recommender(recipe_idx, top_n=5):
    distances, indices = nn_model.kneighbors(X_tfidf[recipe_idx], n_neighbors=top_n + 1)
    
    original_name = df.iloc[recipe_idx]['Name']
    original_cat = df.iloc[recipe_idx].get('RecipeCategory', 'Не указана')
    original_ing = df.iloc[recipe_idx]['ingredients_bert_clean']
    
    print(f"initial recipe: {original_name.upper()}")
    print(f"category: {original_cat}")
    print(f"ingredients (after BERT): {original_ing}\n")
    print("-" * 60)
    print(f" TOP-{top_n} RECOMMENDATIONS:\n")
    
    for i in range(1, len(indices[0])):
        idx = indices[0][i]
        sim_score = 1 - distances[0][i]
        
        name = df.iloc[idx]['Name']
        cat = df.iloc[idx].get('RecipeCategory', 'Не указана')
        ing = df.iloc[idx]['ingredients_bert_clean']
        
        print(f"{i}. {name} (similarity: {sim_score:.2f}) | category: {cat}")
        short_ing = (ing[:80] + '...') if len(ing) > 80 else ing
        print(f"   [ Ingredients: {short_ing} ]\n")


# profile-based recommendations
def recommend_for_user(user_liked_indices, top_n=5):
    user_vector = np.zeros(X_tfidf.shape[1])
    
    for idx in user_liked_indices:
        user_vector += X_tfidf[idx].toarray().ravel()
        
    user_vector = user_vector / len(user_liked_indices)
    

    distances, indices = nn_model.kneighbors(
        user_vector.reshape(1, -1), 
        n_neighbors=top_n + len(user_liked_indices)
    )
    
    print("recommendations for user\n")
    print("Liked recipes:")
    for idx in user_liked_indices:
        print(f" • {df.iloc[idx]['Name']} ({df.iloc[idx].get('RecipeCategory', '')})")
    print("-" * 60)
    
    count = 0
    for i in range(len(indices[0])):
        idx = indices[0][i]

        if idx in user_liked_indices:
            continue
            
        sim_score = 1 - distances[0][i]
        name = df.iloc[idx]['Name']
        cat = df.iloc[idx].get('RecipeCategory', 'Не указана')
        ing = df.iloc[idx]['ingredients_bert_clean']
        
        count += 1
        print(f"{count}. {name} (similarity with profile: {sim_score:.2f}) | category: {cat}")
        short_ing = (ing[:80] + '...') if len(ing) > 80 else ing
        print(f"   [ Ingredients: {short_ing} ]\n")
        
        if count == top_n:
            break

In [19]:
test_recommender(recipe_idx=20)

initial recipe: LOW-FAT BURGUNDY BEEF & VEGETABLE STEW
category: Stew
ingredients (after BERT): beef eye round dried thyme leaves salt pepper ready-to-serve beef broth Burgundy wine garlic cornstarch frozen sugar snap peas

------------------------------------------------------------
 TOP-5 RECOMMENDATIONS:

1. A Perfect Eye of Round Roast Beef (similarity: 0.41) | category: < 4 Hours
   [ Ingredients: beef eye round salt pepper paprika ]

2. Roy Rogers Regular Roast Beef Sandwich (similarity: 0.40) | category: Roast Beef
   [ Ingredients: beef eye round beef broth barbecue sauce ]

3. Easiest Swiss Steak (similarity: 0.39) | category: Steak
   [ Ingredients: beef eye round flour beef broth ]

4. Melt in Your Mouth Eye of Round Beef Roast (similarity: 0.38) | category: Meat
   [ Ingredients: beef eye round sea salt black pepper ]

5. Weight Watchers Spicy Pot Roast (similarity: 0.36) | category: Meat
   [ Ingredients: beef eye round ground coriander salt pepper ground ginger cinnamon c

In [20]:
user_profile_vector = np.zeros(X_baseline.shape[1])

DECAY_FACTOR = 0.9

for idx in user_liked_indices:
    recipe_vector = X_baseline[idx].toarray().ravel()
    user_profile_vector *= DECAY_FACTOR
    user_profile_vector += recipe_vector

#profile normalization (optional)
user_profile_vector = user_profile_vector / len(user_liked_indices)

# 5. Вывод рекомендаций
print("liked recipes:")
for idx in user_liked_indices:
    print(f" - {df.iloc[idx]['Name']}")
print("=" * 60)

liked recipes:
 - Oatmeal Raisin Cookies
 - Spiced Peanut &amp; Sweet Potato Soup
 - Hawaiian Papaya Chutney
 - Fruit Punch-Non Alcoholic
 - Chicken Parmesan
 - Southwest Guacamole
 - Outback Steakhouse's Dipping Sauce
 - Beef Steak Seasoning Mix
 - Continental Chicken
 - Curried Spinach Sauce
 - Curried Spinach Sauce
 - Cream Cheese Pound Cake
 - Caramel Sultana Square
 - Alaskan Curried Shrimp


In [21]:
user_liked_indices = [
    3735,   
    454645,   
    1284, 
    45647, 
    45435,  
    5007,  
    8000,  
    78906, 
    165870, 
    543,
    543,
    78664,
    98,
    7654 
]
recommend_for_user(user_liked_indices, top_n=5)

recommendations for user

Liked recipes:
 • Oatmeal Raisin Cookies (Drop Cookies)
 • Spiced Peanut &amp; Sweet Potato Soup (< 60 Mins)
 • Hawaiian Papaya Chutney (Chutneys)
 • Fruit Punch-Non Alcoholic (Punch Beverage)
 • Chicken Parmesan (Chicken)
 • Southwest Guacamole (Fruit)
 • Outback Steakhouse's Dipping Sauce (Weeknight)
 • Beef Steak Seasoning Mix (Low Cholesterol)
 • Continental Chicken (One Dish Meal)
 • Curried Spinach Sauce (Vegetable)
 • Curried Spinach Sauce (Vegetable)
 • Cream Cheese Pound Cake (Dessert)
 • Caramel Sultana Square (Dessert)
 • Alaskan Curried Shrimp (Lunch/Snacks)
------------------------------------------------------------
1. Rib Rub (similarity with profile: 0.31) | category: Low Cholesterol
   [ Ingredients: brown sugar paprika salt black pepper white pepper cayenne pepper garlic powder ... ]

2. Barbecue Potatoes (Oven or Grill) (similarity with profile: 0.31) | category: Potato
   [ Ingredients: chili powder onion powder garlic powder sugar paprika 

In [22]:
df.iloc[5435]

RecipeId                                                               8401
Name                                    Stuffed Any Way You Want Hamburgers
Images                                                         character(0)
RecipeCategory                                                         Meat
Keywords                                                        "< 15 Mins"
RecipeIngredientParts     c("beef", "onion", "salt", "pepper", "egg", "W...
AggregatedRating                                                        4.5
ReviewCount                                                             4.0
TotalTime                                                              PT0S
Calories                                                              805.7
RecipeInstructions        c("Mix all ingredients except the fillings.", ...
ingredients_clean         beef onion salt pepper egg Worcestershire sauc...
keywords_clean                                                    < 15 Mins
features_tex

In [23]:
save_npz('data/X_tfidf.npz', X_tfidf)
joblib.dump(tfidf, 'data/tfidf_vectorizer.pkl')

['data/tfidf_vectorizer.pkl']